## Experiment to identify memory capacity in the RNN hidden state

### Recipe: experiment 1
- generate completely random sequence
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

### Recipe: experiment 2
- generate a sequence with pattern (1,2,3,4,5, 1,2,3,4,5, 1,2,3,4,5....)
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

In [85]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 

In [104]:
def get_patterned_sequence(total_samples, token_number=7):
    sequence = []
    for i in range(total_samples):
        sequence.append(chr((i % token_number) + ord('A')))
    return sequence

In [86]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [ ]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-64
      
            self.y[ii] = \
                ord(data[ii+jj+1])-64

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [88]:
# tokens start from 1
class Dataset_reconstructer(Dataset):
    def __init__(self, hidden_states, data, past_recall=1, short_term_memory=1):
        
        self.X = np.array(hidden_states)
        self.y = np.vectorize(ord)(data) - 64
        self.short_term_memory = short_term_memory
        self.past_recall = past_recall

        if short_term_memory == 1:
            self.y = np.concatenate(
                    (np.zeros(past_recall, dtype=int), self.y[:-past_recall])
                )

        self.X = tnsr(self.X)
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index+self.short_term_memory-self.past_recall-1]

    def __len__(self):
        return self.X.shape[0]

In [89]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='relu')
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        out = self.linear(out[:,-1,:])

        return out, h  

In [90]:
class classifier(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.linear1 = nn.Linear(hidden_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        out = nn.functional.relu(self.linear1(x))
        out = self.linear2(x)

        return out  

In [98]:

reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 50000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 1e-3
repitition = []
acc = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95, weight_decay=1e-8)
        criterion = torch.nn.CrossEntropyLoss()

        data = get_random_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, h = model(X)
            else:
                y_pred, h = model(X, hidden)

            loss = criterion(y_pred, y[0])     
            loss.backward()
            optimizer.step()

            # print(total)
            with torch.no_grad():
                hidden = h.clone()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0


                test_acc = np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                
                
                if total%10000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model(X)
                else:
                    y_pred, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_random.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 10001, loss: 1.7885, accuracy: 0.1440
Iter : 20001, loss: 1.9640, accuracy: 0.1490
Iter : 30001, loss: 1.9064, accuracy: 0.1490
Iter : 40001, loss: 2.0819, accuracy: 0.1590
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:42<00:00,  4.29s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.7462847770866252
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:42<00:00,  4.30s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.1421485289117347
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 0.1459521094640821
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:41<00:00,  4.16s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.14433175298565684
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:41<00:00,  4.16s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.1432115065314369


In [105]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 50000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 1e-3
repitition = []
acc = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95, weight_decay=1e-8)
        criterion = torch.nn.CrossEntropyLoss()

        data = get_patterned_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, h = model(X)
            else:
                y_pred, h = model(X, hidden)

            loss = criterion(y_pred, y[0])     
            loss.backward()
            optimizer.step()

            # print(total)
            with torch.no_grad():
                hidden = h.clone()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0


                test_acc = np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                
                
                if total%10000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model(X)
                else:
                    y_pred, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_patterned.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 10001, loss: 0.0000, accuracy: 1.0000
Iter : 20001, loss: 0.0000, accuracy: 1.0000
Iter : 30001, loss: 0.0000, accuracy: 1.0000
Iter : 40001, loss: 0.0000, accuracy: 1.0000
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:42<00:00,  4.25s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.999879992799568
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:42<00:00,  4.24s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.999839990399424
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 0.0
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:41<00:00,  4.15s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.0
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:40<00:00,  4.08s/it]


Evaluating reconstruction ...
Totoal accuracy : 0.0


In [109]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 50000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 1e-3
repitition = []
acc = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.95, weight_decay=1e-8)
        criterion = torch.nn.CrossEntropyLoss()

        data = get_sequence(total_samples, 2, 3, train_percent=1.0)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, h = model(X)
            else:
                y_pred, h = model(X, hidden)

            loss = criterion(y_pred, y[0])     
            loss.backward()
            optimizer.step()

            # print(total)
            with torch.no_grad():
                hidden = h.clone()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0


                test_acc = np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                
                
                if total%10000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model(X)
                else:
                    y_pred, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        data_ = []
        for ch in data:
            data_.append(ch)

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data_, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Total accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_hard_patterned.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 10001, loss: 0.4785, accuracy: 0.6810
Iter : 20001, loss: 0.6292, accuracy: 0.6530
Iter : 30001, loss: 0.9793, accuracy: 0.6660
Iter : 40001, loss: 1.2958, accuracy: 0.6840
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:42<00:00,  4.29s/it]


Evaluating reconstruction ...
Total accuracy : 0.9668580114806888
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:42<00:00,  4.23s/it]


Evaluating reconstruction ...
Total accuracy : 0.800948056883413
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Total accuracy : 0.47420432495148934
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:41<00:00,  4.18s/it]


Evaluating reconstruction ...
Total accuracy : 0.4041689171617756
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:41<00:00,  4.14s/it]


Evaluating reconstruction ...
Total accuracy : 0.3944267738902559
